In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.


In [2]:
# 2. Configuration
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/NL2SQL")

# Updated ZIP created for the rules-integrated pipeline.
PIPELINE_ZIP = DRIVE_ROOT / "nl2sql_pipeline_hidden_reference_db_id_no_heading.zip"

# Point this at your EXISTING trained DB-ID predictor directory (no training).
DB_MODEL_PATH = Path("/content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor")

# Spider dataset paths. Adjust SPIDER_ROOT if your Drive folder differs.
SPIDER_ROOT = Path("/content/drive/MyDrive/spider_data/spider_data")
SPIDER_TABLES_PATH = SPIDER_ROOT / "tables.json"
SPIDER_DEV_PATH = SPIDER_ROOT / "dev.json"
SPIDER_TRAIN_PATH = SPIDER_ROOT / "train_spider.json"
SPIDER_DB_PATH = SPIDER_ROOT / "database"

PROJECT_DIR = Path("/content/nl2sql_pipeline")
RESULTS_DIR = DRIVE_ROOT / "evaluation_results"

TOP_K = 5
MAX_RETRIES = 3

# Set to 25 or 100 for a quick test; use None for the full Spider dev set.
MAX_EVAL_EXAMPLES = None

print("Pipeline ZIP:", PIPELINE_ZIP)
print("DB model:", DB_MODEL_PATH)
print("Spider root:", SPIDER_ROOT)


Pipeline ZIP: /content/drive/MyDrive/NL2SQL/nl2sql_pipeline_hidden_reference_db_id_no_heading.zip
DB model: /content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor
Spider root: /content/drive/MyDrive/spider_data/spider_data


In [3]:
# 3. Verify required Drive files/folders before installing anything
required = {
    "Pipeline ZIP": PIPELINE_ZIP,
    "DB model directory": DB_MODEL_PATH,
    "Spider tables.json": SPIDER_TABLES_PATH,
    "Spider dev.json": SPIDER_DEV_PATH,
    "Spider database directory": SPIDER_DB_PATH,
}

for name, path in required.items():
    print(f"{name:24} -> {'OK' if path.exists() else 'MISSING'}")
    if path.exists():
        print("   ", path)

missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required items: " + ", ".join(missing) +
        "\nFix the Configuration cell and run this cell again."
    )

model_files = list(DB_MODEL_PATH.rglob("*"))
print(f"\nModel directory contains {len(model_files)} files/items.")
for p in model_files[:15]:
    print(" ", p.relative_to(DB_MODEL_PATH))


Pipeline ZIP             -> OK
    /content/drive/MyDrive/NL2SQL/nl2sql_pipeline_hidden_reference_db_id_no_heading.zip
DB model directory       -> OK
    /content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor
Spider tables.json       -> OK
    /content/drive/MyDrive/spider_data/spider_data/tables.json
Spider dev.json          -> OK
    /content/drive/MyDrive/spider_data/spider_data/dev.json
Spider database directory -> OK
    /content/drive/MyDrive/spider_data/spider_data/database

Model directory contains 11 files/items.
  config_sentence_transformers.json
  config.json
  model.safetensors
  tokenizer_config.json
  tokenizer.json
  sentence_bert_config.json
  1_Pooling
  2_Normalize
  modules.json
  README.md
  1_Pooling/config.json


In [4]:
# 4. Install inference dependencies (no training or fine-tuning)
# Keep the Colab-provided PyTorch version; install the packages used by the pipeline.
!pip -q install -U "langchain>=1.0.0,<2.0.0" langchain-openai "sentence-transformers>=5.0.0,<6.0.0" "datasets>=3.0.0,<5.0.0" accelerate rich python-dotenv

import sentence_transformers, datasets, langchain_openai, langchain
print("langchain:", langchain.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("datasets:", datasets.__version__)
print("langchain-openai:", getattr(langchain_openai, "__version__", "installed"))


langchain: 1.5.0
sentence-transformers: 5.7.0
datasets: 4.8.5
langchain-openai: 1.8.0


In [5]:
# 5. Verify runtime + GPU
import sys, platform
import numpy as np
import torch
import transformers
import sentence_transformers

print("Python:", sys.version)
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("WARNING: CUDA is not available. The pipeline can still run, but DB-ID inference will use CPU.")

# The current DB predictor chooses CUDA automatically when available.
os.environ["DB_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
NumPy: 2.4.6
PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Sentence Transformers: 5.7.0
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0
GPU memory (GB): 14.56


In [6]:
# 6. Extract the updated pipeline ZIP
import shutil, zipfile

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PIPELINE_ZIP, "r") as z:
    z.extractall(PROJECT_DIR)

# The updated ZIP stores src/ and db_rules.json at its root.
children = list(PROJECT_DIR.iterdir())
if len(children) == 1 and children[0].is_dir() and not (PROJECT_DIR / "src").exists():
    ACTUAL_PROJECT_DIR = children[0]
else:
    ACTUAL_PROJECT_DIR = PROJECT_DIR

rules_file = ACTUAL_PROJECT_DIR / "db_rules.json"
if not (ACTUAL_PROJECT_DIR / "src").is_dir():
    raise FileNotFoundError(f"Could not find src/ in extracted project: {ACTUAL_PROJECT_DIR}")
if not rules_file.is_file():
    raise FileNotFoundError(f"Updated ZIP is missing db_rules.json: {rules_file}")

print("Runnable project:", ACTUAL_PROJECT_DIR)
print("src exists:", (ACTUAL_PROJECT_DIR / "src").exists())
print("db_rules.json exists:", rules_file.exists())
print("Rule file size:", f"{rules_file.stat().st_size:,}", "bytes")


Runnable project: /content/nl2sql_pipeline/nl2sql_rules_integrated
src exists: True
db_rules.json exists: True
Rule file size: 252,487 bytes


In [7]:
from google.colab import userdata

In [8]:
# 7. Configure environment variables BEFORE importing project modules
import os, sys, json
from google.colab import userdata

os.environ["SPIDER_TABLES_PATH"] = str(SPIDER_TABLES_PATH)
os.environ["SPIDER_DB_PATH"] = str(SPIDER_DB_PATH)
os.environ["DB_MODEL_PATH"] = str(DB_MODEL_PATH)
os.environ["DB_RULES_PATH"] = str(ACTUAL_PROJECT_DIR / "db_rules.json")
os.environ["DB_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"

# Load the key from Colab Secrets without ever printing the secret.
if not os.getenv("OPENAI_API_KEY"):
    try:
        secret_key = userdata.get("OPENAI_API_KEY")
        if secret_key:
            os.environ["OPENAI_API_KEY"] = secret_key
    except Exception:
        pass

print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
os.chdir(ACTUAL_PROJECT_DIR)
if str(ACTUAL_PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(ACTUAL_PROJECT_DIR))

print("Working directory:", os.getcwd())
print("SPIDER_TABLES_PATH:", os.environ["SPIDER_TABLES_PATH"])
print("SPIDER_DB_PATH:", os.environ["SPIDER_DB_PATH"])
print("DB_MODEL_PATH:", os.environ["DB_MODEL_PATH"])
print("DB_RULES_PATH:", os.environ["DB_RULES_PATH"])
print("DB_DEVICE:", os.environ["DB_DEVICE"])

with open(os.environ["DB_RULES_PATH"], "r", encoding="utf-8") as f:
    db_rules = json.load(f)
print("Database rule entries:", len(db_rules))


OPENAI_API_KEY present: True
Working directory: /content/nl2sql_pipeline/nl2sql_rules_integrated
SPIDER_TABLES_PATH: /content/drive/MyDrive/spider_data/spider_data/tables.json
SPIDER_DB_PATH: /content/drive/MyDrive/spider_data/spider_data/database
DB_MODEL_PATH: /content/drive/MyDrive/nl2sql_pipeline/models/db_id_predictor
DB_RULES_PATH: /content/nl2sql_pipeline/nl2sql_rules_integrated/db_rules.json
DB_DEVICE: cuda
Database rule entries: 166


In [9]:
# 8. Load the trained DB-ID predictor and verify it
from src.db_prediction.model import DBIDPredictor

predictor = DBIDPredictor(
    model_path=str(DB_MODEL_PATH),
    tables_path=str(SPIDER_TABLES_PATH),
    device=os.environ["DB_DEVICE"],
)

print("DB-ID model loaded successfully.")
print("Number of Spider databases in catalog:", len(predictor.db_ids))
print("Embedding matrix shape:", predictor.schema_embeddings.shape)
print("Device:", predictor.device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

DB-ID model loaded successfully.
Number of Spider databases in catalog: 166
Embedding matrix shape: (166, 768)
Device: cuda


In [10]:
SMOKE_TEST_QUESTION = input("Enter NLQ")

Enter NLQCount the number of paragraphs.


In [11]:
# 9. DB-ID retrieval smoke test
question = SMOKE_TEST_QUESTION

candidates = predictor.predict(question, top_k=TOP_K)

print("\nQuestion:", question)
print("\nTop database candidates:")
for c in candidates:
    print(f"{c['rank']:2}. {c['db_id']:30} score={c['bi_score']:.6f}")



Question: Count the number of paragraphs.

Top database candidates:
 1. cre_Doc_Template_Mgt           score=0.536848
 2. cre_Doc_Control_Systems        score=0.292574
 3. cre_Docs_and_Epenses           score=0.289550
 4. cre_Doc_Tracking_DB            score=0.250042
 5. document_management            score=0.238698


In [12]:
# 10. Run the complete NLQ -> SQL pipeline for one question
# Requires GROQ_API_KEY in Colab Secrets or the environment. The key is never printed.
import os
from google.colab import userdata

if not os.getenv("OPENAI_API_KEY"):
    try:
        secret_key = userdata.get("OPENAI_API_KEY")
        if secret_key:
            os.environ["OPENAI_API_KEY"] = secret_key
    except Exception:
        pass

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it in Colab Secrets (🔑) and enable notebook access."
    )

import src.pipeline as pipeline_module
# Reuse the already-loaded predictor. Without this, the pipeline rebuilds all schema
# embeddings for every question, which makes full Spider evaluation unnecessarily slow.
pipeline_module.DBIDPredictor = lambda: predictor
run_nlq_to_sql = pipeline_module.run_nlq_to_sql

# Enable reference-comparison feedback for an exact Spider dev question.
# The reference is used only for this evaluation/demo workflow.
with open(SPIDER_DEV_PATH, "r", encoding="utf-8") as f:
    _dev_for_smoke = json.load(f)
_smoke_matches = [x for x in _dev_for_smoke if x.get("question") == SMOKE_TEST_QUESTION]
_smoke_gold = _smoke_matches[0] if _smoke_matches else None

smoke_kwargs = {
    "question": SMOKE_TEST_QUESTION,
    "top_k": TOP_K,
    "max_retries": MAX_RETRIES,
}
if _smoke_gold is not None:
    smoke_kwargs["gold_db_id"] = _smoke_gold["db_id"]
    smoke_kwargs["gold_sql"] = _smoke_gold["query"]
    print("Reference-guided database/SQL feedback: ENABLED (exact Spider dev match)")
else:
    print("Reference-guided feedback: DISABLED (question not found exactly in Spider dev.json)")

smoke_result = run_nlq_to_sql(**smoke_kwargs)

print("\\n" + "="*80)
print("SMOKE TEST RESULT")
print("="*80)
print("Question:", SMOKE_TEST_QUESTION)
print("Bi-Encoder Top-1:", smoke_result.get("db_id_prediction"))
print("Bi-Encoder Top-1 score:", smoke_result.get("db_prediction_score"))
print("Schema Agent DB:", smoke_result.get("db_id"))
print("Status:", smoke_result.get("status"))
print("Retries:", smoke_result.get("retry_count"))
print("\\nFinal SQL:")
print(smoke_result.get("final_sql"))
print("\\nExecution:")
print(smoke_result.get("execution"))


Reference-guided database/SQL feedback: ENABLED (exact Spider dev match)


1. ORCHESTRATOR

{
    'intent': 'Count the number of paragraphs',
    'feasibility': 'FEASIBLE',
    'complexity': 'easy',
    'requires_join': False,
    'requires_group_by': False,
    'requires_subquery': False,
    'plan': ['Identify the table containing paragraphs', 'Count the rows in the paragraph table']
}

2. BI-ENCODER — TOP DATABASE RETRIEVAL

1. cre_Doc_Template_Mgt           score=0.536848

2. cre_Doc_Control_Systems        score=0.292574

... (10 outputs left out)

{
    'database_match': True,
    'predicted_db_id': 'cre_Doc_Template_Mgt',
    'reference_execution': {'status': 'success', 'columns': ['count(*)'], 'rows': [(15,)], 'row_count': 1},
    'predicted_execution': {'status': 'success', 'columns': ['COUNT(*)'], 'rows': [(15,)], 'row_count': 1},
    'comparison': {'match': True, 'comparison_type': 'UNORDERED_RESULT_MATCH', 'reason': 'Result sets match.'}
}

6. CRITIQUE AGENT

{
    'status': 'PASS',
    'syntax': 'PASS',
    'schema': 'PASS',
    'semantic': 'PASS',
    'execution': 'PASS',
    'database_fit': 'PASS',
    'reason': "The SQL query correctly counts the number of paragraphs in the 'Paragraphs' table, which is directly
relevant to the question about counting paragraphs. All aspects of the query align with the database's purpose and 
schema. Predicted and reference execution result sets match.",
    'correction': '',
    'failure_type': '',
    'retry_action': 'NONE',
    'feedback_to_schema_agent': {}
}

\n================================================================================
SMOKE TEST RESULT
Question: Count the number of paragraphs.
Bi-Encoder Top-1: cre_Doc_Template_Mgt
Bi-Encoder Top-1 score: 0.5368484258651733
Schema Agent DB: cre_Doc_Template_Mgt
Status: PASS
Retries: 0
\nFinal SQL:
SELECT COUNT(*) FROM Paragraphs;
\nExecution:
{'status': 'success', 'columns': ['COUNT(*)'], 'rows': [(15,)], 'row_count': 1}


### Force Refresh Groq API Key
Run the cell below to clear any cached environment keys and reload your new `GROQ_API_KEY` directly from Colab Secrets.

In [13]:
def execute_on_db(db_id, sql):
    from src.tools.execute_sql import execute_sql_raw
    return execute_sql_raw(db_id, sql)

In [14]:
def normalize_rows(rows):
    # Preserve duplicate rows while making values comparable.
    return sorted([tuple(str(v) for v in row) for row in rows])

In [15]:
# 10A. Professor Demo — final SQL vs gold SQL
import json
from IPython.display import display

# Use the question entered in the smoke-test cell.
demo_question = SMOKE_TEST_QUESTION

# Ensure spider_dev is loaded
if 'spider_dev' not in globals():
    with open(SPIDER_DEV_PATH, "r", encoding="utf-8") as f:
        spider_dev = json.load(f)

# Find the corresponding Spider gold example.
matches = [x for x in spider_dev if x.get("question") == demo_question]

if not matches:
    raise ValueError(
        "This question was not found in dev.json. "
        "For a professor demo, enter an exact Spider dev question in the input cell."
    )

gold_item = matches[0]
gold_db_id = gold_item["db_id"]
gold_sql = gold_item["query"]

# Use the already-generated pipeline result.
demo_result = smoke_result
predicted_db_id = demo_result.get("db_id")
final_sql = demo_result.get("final_sql") or ""

if not predicted_db_id:
    raise RuntimeError("The pipeline did not return a predicted DB ID.")
if not final_sql:
    raise RuntimeError("The pipeline did not return a final SQL query.")

# Execute predicted SQL on the DB selected by the pipeline.
pred_exec = execute_on_db(predicted_db_id, final_sql)

# Execute gold SQL on the gold Spider DB.
gold_exec = execute_on_db(gold_db_id, gold_sql)

def demo_rows(result):
    if result.get("status") != "success":
        return None
    return normalize_rows(result.get("rows", []))

pred_rows = demo_rows(pred_exec)
gold_rows = demo_rows(gold_exec)

execution_correct = (
    pred_exec.get("status") == "success"
    and gold_exec.get("status") == "success"
    and pred_rows == gold_rows
)

print("=" * 100)
print("SPIDER EXECUTION ACCURACY — SINGLE QUESTION DEMO")
print("=" * 100)

print("\nQuestion:")
print(demo_question)

print("\n" + "-" * 100)
print("DATABASES")
print("-" * 100)
print("Gold DB     :", gold_db_id)
print("Predicted DB:", predicted_db_id)
print("DB Match    :", "YES" if predicted_db_id == gold_db_id else "NO")

print("\n" + "-" * 100)
print("FINAL SQL GENERATED BY PIPELINE")
print("-" * 100)
print(final_sql)

print("\n" + "-" * 100)
print("GOLD SQL FROM SPIDER")
print("-" * 100)
print(gold_sql)

print("\n" + "-" * 100)
print("PREDICTED SQL EXECUTION RESULT")
print("-" * 100)
print("Status:", pred_exec.get("status"))
print("Rows:")
if pred_exec.get("status") == "success":
    display(pred_exec.get("rows", []))
else:
    print(pred_exec.get("error", pred_exec))

print("\n" + "-" * 100)
print("GOLD SQL EXECUTION RESULT")
print("-" * 100)
print("Status:", gold_exec.get("status"))
print("Rows:")
if gold_exec.get("status") == "success":
    display(gold_exec.get("rows", []))
else:
    print(gold_exec.get("error", gold_exec))

print("\n" + "=" * 100)
print("EXECUTION ACCURACY:", "PASS ✓" if execution_correct else "FAIL ✗")
print("=" * 100)

if execution_correct:
    print("The final SQL generated by the pipeline produced the same result set as the Spider gold SQL.")
else:
    print("The generated SQL did not produce the same result set as the Spider gold SQL.")


SPIDER EXECUTION ACCURACY — SINGLE QUESTION DEMO

Question:
Count the number of paragraphs.

----------------------------------------------------------------------------------------------------
DATABASES
----------------------------------------------------------------------------------------------------
Gold DB     : cre_Doc_Template_Mgt
Predicted DB: cre_Doc_Template_Mgt
DB Match    : YES

----------------------------------------------------------------------------------------------------
FINAL SQL GENERATED BY PIPELINE
----------------------------------------------------------------------------------------------------
SELECT COUNT(*) FROM Paragraphs;

----------------------------------------------------------------------------------------------------
GOLD SQL FROM SPIDER
----------------------------------------------------------------------------------------------------
SELECT count(*) FROM Paragraphs

----------------------------------------------------------------------------------

[(15,)]


----------------------------------------------------------------------------------------------------
GOLD SQL EXECUTION RESULT
----------------------------------------------------------------------------------------------------
Status: success
Rows:


[(15,)]


EXECUTION ACCURACY: PASS ✓
The final SQL generated by the pipeline produced the same result set as the Spider gold SQL.


In [ ]:
# print(spider_dev)
import pandas as pd
dev_data = pd.DataFrame(spider_dev)
dev_data.head()
dev_data.to_csv('dev_data.csv', index=False)

## 11. Spider evaluation

This notebook reports three separate measurements:

1. **DB retrieval:** Top-1 accuracy and Top-3/5/10 recall from the trained DB-ID predictor.
2. **End-to-end NLQ-only execution accuracy:** SQL generated by the pipeline is executed on the **selected/predicted database** and compared with the gold SQL result executed on the **gold database**.
3. **Oracle-DB execution accuracy:** Generated SQL is executed on the gold database, isolating SQL generation from database-selection errors.

The end-to-end comparison deliberately executes the gold query on the gold DB, not on the predicted DB. That avoids incorrectly comparing a result from one database against a gold query run on another database.

No DB-ID training, rule generation, or fine-tuning is performed.


In [ ]:
# @title
# 12. Build a robust NLQ-only Spider evaluator
import json, csv, time, traceback
from pathlib import Path

with open(SPIDER_DEV_PATH, "r", encoding="utf-8") as f:
    spider_dev = json.load(f)

if MAX_EVAL_EXAMPLES is not None:
    spider_dev = spider_dev[:MAX_EVAL_EXAMPLES]

print("Evaluation examples:", len(spider_dev))

def normalize_rows(rows):
    # Preserve duplicate rows while making row order irrelevant.
    return sorted([tuple(str(v) for v in row) for row in rows])

def execute_on_db(db_id, sql):
    from src.tools.execute_sql import execute_sql_raw
    return execute_sql_raw(db_id, sql)

def result_match(pred_db_id, predicted_sql, gold_db_id, gold_sql):
    if not predicted_sql:
        return False, {"pred": {"status": "empty"}, "gold": {"status": "not_run"}}

    pred = execute_on_db(pred_db_id, predicted_sql)
    gold = execute_on_db(gold_db_id, gold_sql)

    if pred.get("status") != "success" or gold.get("status") != "success":
        return False, {"pred": pred, "gold": gold}

    return (
        normalize_rows(pred.get("rows", [])) == normalize_rows(gold.get("rows", []))
    ), {"pred": pred, "gold": gold}

def retrieval_recall(candidates, gold_db, k):
    return any(c["db_id"] == gold_db for c in candidates[:k])

results = []
start_time = time.time()

for i, item in enumerate(spider_dev, 1):
    q = item["question"]
    gold_db = item["db_id"]
    gold_sql = item["query"]

    record = {
        "index": i,
        "question": q,
        "gold_db_id": gold_db,
        "top1_db": None,
        "selected_db": None,
        "top1_score": None,
        "top1_correct": False,
        "top3_recall": False,
        "top5_recall": False,
        "top10_recall": False,
        "pipeline_status": None,
        "retries": 0,
        "predicted_sql": "",
        "end_to_end_execution_correct": False,
        "oracle_db_execution_correct": False,
        "predicted_db_execution_status": None,
        "oracle_db_execution_status": None,
        "error": None,
    }

    try:
        candidates = predictor.predict(q, top_k=TOP_K)
        record["top1_db"] = candidates[0]["db_id"]
        record["top1_score"] = candidates[0]["bi_score"]
        record["top1_correct"] = record["top1_db"] == gold_db
        record["top3_recall"] = retrieval_recall(candidates, gold_db, 3)
        record["top5_recall"] = retrieval_recall(candidates, gold_db, 5)
        record["top10_recall"] = retrieval_recall(candidates, gold_db, 10)

        # Pipeline performs its own retrieval, then rules-informed schema selection.
        pipeline_result = run_nlq_to_sql(
            question=q,
            top_k=TOP_K,
            max_retries=MAX_RETRIES,
            gold_db_id=gold_db,
            gold_sql=gold_sql,
        )

        record["selected_db"] = pipeline_result.get("db_id")
        record["pipeline_status"] = pipeline_result.get("status")
        record["retries"] = pipeline_result.get("retry_count", 0)
        record["predicted_sql"] = pipeline_result.get("final_sql") or ""

        pred_db = record["selected_db"]
        pred_sql = record["predicted_sql"]

        # True end-to-end: predicted SQL on selected DB vs gold SQL on gold DB.
        if pred_db and pred_sql:
            e2e_correct, pred_details = result_match(pred_db, pred_sql, gold_db, gold_sql)
            record["end_to_end_execution_correct"] = e2e_correct
            record["predicted_db_execution_status"] = pred_details["pred"].get("status")

        # Oracle-DB: execute generated SQL and gold SQL on the gold database.
        if pred_sql:
            oracle_correct, oracle_details = result_match(gold_db, pred_sql, gold_db, gold_sql)
            record["oracle_db_execution_correct"] = oracle_correct
            record["oracle_db_execution_status"] = oracle_details["pred"].get("status")

    except Exception as exc:
        record["error"] = repr(exc)
        print(f"\\nERROR on example {i}: {q}\\n{exc}")

    results.append(record)

    if i % 10 == 0 or i == len(spider_dev):
        elapsed = time.time() - start_time
        done = i
        e2e = sum(r["end_to_end_execution_correct"] for r in results) / done
        top1 = sum(r["top1_correct"] for r in results) / done
        top10 = sum(r["top10_recall"] for r in results) / done
        print(
            f"[{done}/{len(spider_dev)}] "
            f"Top1={top1:.3%} Top10={top10:.3%} "
            f"E2E={e2e:.3%} elapsed={elapsed/60:.1f}m"
        )


In [ ]:
# 13. Summarize evaluation metrics
import pandas as pd

df = pd.DataFrame(results)

def pct(col):
    return float(df[col].mean()) if len(df) else 0.0

metrics = {
    "examples": len(df),
    "db_top1_accuracy": pct("top1_correct"),
    "db_top3_recall": pct("top3_recall"),
    "db_top5_recall": pct("top5_recall"),
    "db_top10_recall": pct("top10_recall"),
    "selected_db_equals_gold": float((df["selected_db"] == df["gold_db_id"]).mean()) if len(df) else 0.0,
    "end_to_end_execution_accuracy": pct("end_to_end_execution_correct"),
    "oracle_db_execution_accuracy": pct("oracle_db_execution_correct"),
    "pipeline_pass_rate": float((df["pipeline_status"] == "PASS").mean()) if len(df) else 0.0,
    "avg_retries": float(df["retries"].mean()) if len(df) else 0.0,
    "errors": int(df["error"].notna().sum()) if len(df) else 0,
}

print("="*80)
print("SPIDER NL2SQL EVALUATION")
print("="*80)
for k, v in metrics.items():
    if isinstance(v, float) and "accuracy" in k or isinstance(v, float) and "recall" in k or k == "pipeline_pass_rate":
        print(f"{k:35}: {v:.3%}")
    else:
        print(f"{k:35}: {v}")

print("\nInterpretation:")
print("- End-to-end execution accuracy = the metric that reflects your NLQ-only pipeline.")
print("- Oracle-DB execution accuracy = SQL-generation quality assuming the correct DB is known.")
print("- The gap between them indicates how much DB selection is costing you.")


In [ ]:
# Check which relevant variables still exist in memory
for name in [
    "eval_df",
    "metrics",
    "results",
    "all_results",
    "evaluation_results",
    "predictions",
]:
    if name in globals():
        value = globals()[name]
        try:
            print(name, type(value).__name__, "length =", len(value))
        except TypeError:
            print(name, type(value).__name__)

metrics dict length = 11
results list length = 100


In [ ]:
# 14. Inspect failures
failure_df = df[
    (~df["end_to_end_execution_correct"]) |
    (df["selected_db"] != df["gold_db_id"])
].copy()

print("Failure/diagnostic cases:", len(failure_df))

display(
    failure_df[
        [
            "index",
            "question",
            "gold_db_id",
            "top1_db",
            "selected_db",
            "pipeline_status",
            "retries",
            "predicted_sql",
            "end_to_end_execution_correct",
            "oracle_db_execution_correct",
            "error",
        ]
    ].head(50)
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 15. Save results permanently to Google Drive
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

json_path = RESULTS_DIR / "spider_nlq_evaluation.json"
csv_path = RESULTS_DIR / "spider_nlq_evaluation.csv"
metrics_path = RESULTS_DIR / "spider_nlq_metrics.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)

df.to_csv(csv_path, index=False)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved:")
print(json_path)
print(csv_path)
print(metrics_path)


Saved:
/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation.json
/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation.csv
/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_metrics.json


In [ ]:
from pathlib import Path

results_dir = Path(
    "/content/drive/MyDrive/NL2SQL/evaluation_results"
)

if results_dir.exists():
    for path in results_dir.rglob("*"):
        if path.is_file():
            print(path, f"({path.stat().st_size:,} bytes)")
else:
    print("Results directory not found.")

/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation.json (54,208 bytes)
/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation.csv (22,543 bytes)
/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_metrics.json (436 bytes)
/content/drive/MyDrive/NL2SQL/evaluation_results/nl2sql_pipeline_with_colab_db_predictor.zip (69,815 bytes)


In [ ]:
import json
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/NL2SQL/evaluation_results"
)

json_path = RESULTS_DIR / "spider_nlq_evaluation.json"
csv_path = RESULTS_DIR / "spider_nlq_evaluation.csv"

with open(json_path, "r") as f:
    saved_data = json.load(f)

print("In-memory results:", len(results))
print("Saved JSON type:", type(saved_data).__name__)

if isinstance(saved_data, list):
    print("Saved JSON results:", len(saved_data))
elif isinstance(saved_data, dict):
    print("Saved JSON keys:", list(saved_data.keys()))
    for key, value in saved_data.items():
        if isinstance(value, list):
            print(f"{key}: {len(value)} entries")

df = pd.read_csv(csv_path)
print("Saved CSV rows:", len(df))
print("CSV columns:", df.columns.tolist())

In [ ]:
print("Last in-memory result:")
print(results[-1] if results else "No results")

print("\nLast few result entries:")
for item in results[-3:]:
    print(item)
    print("-" * 60)

In [ ]:
import json
from pathlib import Path
from collections import Counter

p = Path(
    "/content/drive/MyDrive/NL2SQL/evaluation_results/"
    "spider_nlq_evaluation.json"
)

saved = json.loads(p.read_text())

print("Saved rows:", len(saved))
print("First index:", saved[0].get("index") if saved else None)
print("Last index:", saved[-1].get("index") if saved else None)
print("Unique indices:", len({r.get("index") for r in saved}))
print("Statuses:", Counter(r.get("pipeline_status") for r in saved))

Saved rows: 73
First index: 1
Last index: 73
Unique indices: 73
Statuses: Counter({'PASS': 67, 'FAILED_MAX_RETRIES': 6})


In [ ]:
# Find likely dataset variables
for name, value in list(globals().items()):
    if any(k in name.lower() for k in ["dev", "dataset", "examples", "eval"]):
        try:
            if isinstance(value, (list, tuple)):
                print(name, ":", len(value), "items")
        except Exception:
            pass

_dev_for_smoke : 1034 items
spider_dev : 1034 items


In [ ]:

# RESUME SPIDER EVALUATION FROM GOOGLE DRIVE CHECKPOINT
# Uses the existing pipeline, predictor, retry settings, and saved results.

import json
import csv
import time
import os
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/NL2SQL/evaluation_results"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

JSON_PATH = RESULTS_DIR / "spider_nlq_evaluation.json"
CSV_PATH = RESULTS_DIR / "spider_nlq_evaluation.csv"
METRICS_PATH = RESULTS_DIR / "spider_nlq_metrics.json"
CHECKPOINT_PATH = RESULTS_DIR / "spider_nlq_evaluation_checkpoint.json"

# ---------- 1. Verify the current runtime ----------
required_names = ["run_nlq_to_sql", "predictor", "spider_dev"]
missing = [n for n in required_names if n not in globals()]
if missing:
    raise RuntimeError(
        f"Missing runtime objects: {missing}. "
        "Run the pipeline setup cells first."
    )

if not isinstance(spider_dev, list) or not spider_dev:
    raise RuntimeError("spider_dev is missing or empty.")

# ---------- 2. Load the existing completed results ----------
source_path = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else JSON_PATH

if source_path.exists():
    with open(source_path, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []

if not isinstance(results, list):
    raise TypeError("Saved checkpoint must contain a list of result records.")

# Validate indices and confirm saved questions match the dataset.
by_index = {}
for record in results:
    idx = int(record["index"])
    if idx < 1 or idx > len(spider_dev):
        raise ValueError(f"Invalid saved index: {idx}")
    if idx in by_index:
        raise ValueError(f"Duplicate saved index: {idx}")

    expected_question = spider_dev[idx - 1]["question"]
    if record.get("question") != expected_question:
        raise ValueError(
            f"Saved question does not match Spider dev at index {idx}. "
            "Stopping to protect existing results."
        )
    by_index[idx] = record

results = [by_index[i] for i in sorted(by_index)]
completed = set(by_index)

print(f"Dataset size: {len(spider_dev)}")
print(f"Recovered completed records: {len(completed)}")
print(f"Remaining records: {len(spider_dev) - len(completed)}")

if len(completed) == len(spider_dev):
    print("All examples already have saved records. Nothing to resume.")
else:
    # ---------- 3. Define the existing evaluation helpers ----------
    def normalize_rows(rows):
        return sorted(tuple(str(v) for v in row) for row in rows)

    def execute_on_db(db_id, sql):
        from src.tools.execute_sql import execute_sql_raw
        return execute_sql_raw(db_id, sql)

    def result_match(pred_db_id, predicted_sql, gold_db_id, gold_sql):
        if not predicted_sql:
            return False, {
                "pred": {"status": "empty"},
                "gold": {"status": "not_run"},
            }

        pred = execute_on_db(pred_db_id, predicted_sql)
        gold = execute_on_db(gold_db_id, gold_sql)

        if pred.get("status") != "success" or gold.get("status") != "success":
            return False, {"pred": pred, "gold": gold}

        return (
            normalize_rows(pred.get("rows", []))
            == normalize_rows(gold.get("rows", [])),
            {"pred": pred, "gold": gold},
        )

    def retrieval_recall(candidates, gold_db, k):
        return any(c["db_id"] == gold_db for c in candidates[:k])

    # Write JSON atomically so an interrupted write is less likely to
    # corrupt the last completed checkpoint.
    def save_checkpoint():
        results.sort(key=lambda r: int(r["index"]))
        temp_path = CHECKPOINT_PATH.with_suffix(".tmp")
        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, default=str)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, CHECKPOINT_PATH)

        # Keep the familiar results file updated too.
        temp_json = JSON_PATH.with_suffix(".tmp")
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, default=str)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_json, JSON_PATH)

    start_time = time.time()
    newly_completed = 0

    # ---------- 4. Resume at the first missing index ----------
    for i, item in enumerate(spider_dev, start=1):
        if i in completed:
            continue

        q = item["question"]
        gold_db = item["db_id"]
        gold_sql = item["query"]

        record = {
            "index": i,
            "question": q,
            "gold_db_id": gold_db,
            "top1_db": None,
            "selected_db": None,
            "top1_score": None,
            "top1_correct": False,
            "top3_recall": False,
            "top5_recall": False,
            "top10_recall": False,
            "pipeline_status": None,
            "retries": 0,
            "predicted_sql": "",
            "end_to_end_execution_correct": False,
            "oracle_db_execution_correct": False,
            "predicted_db_execution_status": None,
            "oracle_db_execution_status": None,
            "error": None,
        }

        try:
            candidates = predictor.predict(q, top_k=TOP_K)
            if not candidates:
                raise RuntimeError("DB predictor returned no candidates.")

            record["top1_db"] = candidates[0]["db_id"]
            record["top1_score"] = candidates[0]["bi_score"]
            record["top1_correct"] = record["top1_db"] == gold_db
            record["top3_recall"] = retrieval_recall(candidates, gold_db, 3)
            record["top5_recall"] = retrieval_recall(candidates, gold_db, 5)
            record["top10_recall"] = retrieval_recall(candidates, gold_db, 10)

            pipeline_result = run_nlq_to_sql(
                question=q,
                top_k=TOP_K,
                max_retries=MAX_RETRIES,
                gold_db_id=gold_db,
                gold_sql=gold_sql,
            )

            record["selected_db"] = pipeline_result.get("db_id")
            record["pipeline_status"] = pipeline_result.get("status")
            record["retries"] = pipeline_result.get("retry_count", 0)
            record["predicted_sql"] = pipeline_result.get("final_sql") or ""

            pred_db = record["selected_db"]
            pred_sql = record["predicted_sql"]

            if pred_db and pred_sql:
                e2e_correct, details = result_match(
                    pred_db, pred_sql, gold_db, gold_sql
                )
                record["end_to_end_execution_correct"] = e2e_correct
                record["predicted_db_execution_status"] = (
                    details["pred"].get("status")
                )

            if pred_sql:
                oracle_correct, details = result_match(
                    gold_db, pred_sql, gold_db, gold_sql
                )
                record["oracle_db_execution_correct"] = oracle_correct
                record["oracle_db_execution_status"] = (
                    details["pred"].get("status")
                )

        except Exception as exc:
            record["error"] = repr(exc)
            print(f"\nERROR at index {i}: {q}\n{exc}")

        # Mark this index complete even if evaluation returned an error,
        # so a runtime restart doesn't silently repeat it.
        results.append(record)
        completed.add(i)
        newly_completed += 1

        # Persist every finished question.
        save_checkpoint()

        if newly_completed % 5 == 0 or i == len(spider_dev):
            df_progress = pd.DataFrame(results)
            e2e = (
                df_progress["end_to_end_execution_correct"].mean()
                if len(df_progress) else 0
            )
            top1 = (
                df_progress["top1_correct"].mean()
                if len(df_progress) else 0
            )
            print(
                f"[index {i}/{len(spider_dev)}] "
                f"saved={len(results)} "
                f"new_this_run={newly_completed} "
                f"Top1={top1:.2%} E2E={e2e:.2%} "
                f"elapsed={(time.time()-start_time)/60:.1f} min"
            )

    # ---------- 5. Export final CSV and metrics ----------
    results.sort(key=lambda r: int(r["index"]))
    df = pd.DataFrame(results)

    def pct(col):
        return float(df[col].mean()) if len(df) else 0.0

    metrics = {
        "examples": len(df),
        "db_top1_accuracy": pct("top1_correct"),
        "db_top3_recall": pct("top3_recall"),
        "db_top5_recall": pct("top5_recall"),
        "db_top10_recall": pct("top10_recall"),
        "selected_db_equals_gold": (
            float((df["selected_db"] == df["gold_db_id"]).mean())
            if len(df) else 0.0
        ),
        "end_to_end_execution_accuracy": pct(
            "end_to_end_execution_correct"
        ),
        "oracle_db_execution_accuracy": pct(
            "oracle_db_execution_correct"
        ),
        "pipeline_pass_rate": (
            float((df["pipeline_status"] == "PASS").mean())
            if len(df) else 0.0
        ),
        "avg_retries": float(df["retries"].mean()) if len(df) else 0.0,
        "errors": int(df["error"].notna().sum()) if len(df) else 0,
    }

    df.to_csv(CSV_PATH, index=False)
    with open(METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print("\nRESUMED EVALUATION FINISHED")
    print("Total saved:", len(results))
    print("Checkpoint:", CHECKPOINT_PATH)
    print("JSON:", JSON_PATH)
    print("CSV:", CSV_PATH)
    print("Metrics:", METRICS_PATH)
    print(json.dumps(metrics, indent=2))


In [ ]:
import json
import pandas as pd
from pathlib import Path

# Update this path if your JSON is stored elsewhere
JSON_PATH = "/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

df = pd.DataFrame(results)

required = ["end_to_end_execution_correct", "predicted_db_execution_status"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise KeyError(f"Missing required keys: {missing}")

# Execution accuracy: correct end-to-end results / total evaluated examples
df["end_to_end_execution_correct"] = (
    df["end_to_end_execution_correct"].astype("boolean")
)

total = len(df)
correct = int((df["end_to_end_execution_correct"] == True).sum())
execution_accuracy = (correct / total * 100) if total else 0.0

# Execution success rate: successful SQL executions / total examples
successful = int(
    df["predicted_db_execution_status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("success")
    .sum()
)
execution_success_rate = (successful / total * 100) if total else 0.0

print(f"Total examples: {total}")
print(f"End-to-end correct: {correct}")
print(f"Execution Accuracy: {execution_accuracy:.2f}%")
print(f"SQL Execution Successes: {successful}")
print(f"SQL Execution Success Rate: {execution_success_rate:.2f}%")

Total examples: 100
End-to-end correct: 87
Execution Accuracy: 87.00%
SQL Execution Successes: 98
SQL Execution Success Rate: 98.00%


In [ ]:
from src.pipeline import run_nlq_to_sql
import inspect

print("Pipeline function imported successfully.")
print("Signature:", inspect.signature(run_nlq_to_sql))

Pipeline function imported successfully.
Signature: (question: str, max_retries: int = 3, top_k: int = 10, gold_db_id: str | None = None, gold_sql: str | None = None) -> dict


In [ ]:
# FULL PIPELINE RETRY — failed NLQs only
# Generates NEW SQL; does not merely re-execute the old predicted_sql.

import json
import inspect
from pathlib import Path
from tqdm.auto import tqdm

FAILED_MAX_RETRIES = 3

RESULTS_PATH = Path(
    "/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation.json"
)

# Load saved evaluation results
with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

# Locate failed NLQs
def is_failed(item):
    return (
        item.get("end_to_end_execution_correct") is not True
        or str(item.get("predicted_db_execution_status", "")).lower() != "success"
    )

failed_indices = [
    i for i, item in enumerate(results)
    if is_failed(item)
]

print(f"Total NLQs: {len(results)}")
print(f"Failed NLQs: {len(failed_indices)}")
print(f"Maximum additional full-pipeline attempts: {FAILED_MAX_RETRIES}")
print("Pipeline function:", inspect.signature(run_nlq_to_sql))

for i in tqdm(failed_indices, desc="Full pipeline retries"):
    original = results[i]
    question = original["question"]

    for attempt in range(1, FAILED_MAX_RETRIES + 1):
        if not is_failed(results[i]):
            break

        print(
            f"\nNLQ index={original.get('index', i)} "
            f"| Full-pipeline attempt {attempt}/{FAILED_MAX_RETRIES}"
        )
        print("Question:", question)

        try:
            # Re-run the full NLQ -> database/schema retrieval -> SQL generation
            # -> validation/execution -> feedback/repair pipeline.
            fresh = run_nlq_to_sql(
                question=question,
                gold_db_id=original["gold_db_id"],
                gold_sql=original.get("gold_sql"),
            )

            if not isinstance(fresh, dict):
                raise TypeError(
                    "run_nlq_to_sql() must return a dictionary for this cell. "
                    f"Received: {type(fresh).__name__}"
                )

            # Preserve the original evaluation metadata, but replace prediction
            # and pipeline-result fields with the newly generated result.
            prediction_fields = [
                "selected_db", "top1_db", "top1_score",
                "pipeline_status", "retries", "predicted_sql",
                "end_to_end_execution_correct",
                "oracle_db_execution_correct",
                "predicted_db_execution_status",
                "oracle_db_execution_status", "error",
            ]

            for key in prediction_fields:
                if key in fresh:
                    original[key] = fresh[key]

            original["full_pipeline_retry_attempts"] = (
                original.get("full_pipeline_retry_attempts", 0) + 1
            )

            print("New SQL:", original.get("predicted_sql"))
            print("Execution status:",
                  original.get("predicted_db_execution_status"))
            print("End-to-end correct:",
                  original.get("end_to_end_execution_correct"))

        except Exception as e:
            original["last_full_pipeline_retry_error"] = repr(e)
            print(f"Retry failed with exception: {e}")

        # Save progress after every attempt
        with open(RESULTS_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

print("\nFull-pipeline retry pass completed.")
print("Remaining failed NLQs:", sum(is_failed(x) for x in results))
print("Updated results:", RESULTS_PATH)

In [ ]:

from google.colab import drive
import json
import shutil
from pathlib import Path



# 2. Set the path to your original JSON file in Drive
file_path = Path(
    "/content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation.json"
)

# 3. Create a backup before modifying the original
backup_path = file_path.with_name(
    file_path.stem + "_backup" + file_path.suffix
)
shutil.copy2(file_path, backup_path)

# 4. Keys to remove
keys_to_remove = {
    "oracle_db_execution_correct",
    "predicted_db_execution_status",
    "oracle_db_execution_status",
}

# 5. Load the original JSON
with file_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

# 6. Remove these keys recursively from all JSON objects
def remove_keys(obj):
    if isinstance(obj, dict):
        for key in keys_to_remove:
            obj.pop(key, None)

        for value in obj.values():
            remove_keys(value)

    elif isinstance(obj, list):
        for item in obj:
            remove_keys(item)

remove_keys(data)

# 7. Replace the original JSON file
with file_path.open("w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
    f.write("\n")

print("Successfully updated the original JSON file!")
print(f"Updated file: {file_path}")
print(f"Backup saved: {backup_path}")


Successfully updated the original JSON file!
Updated file: /content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation.json
Backup saved: /content/drive/MyDrive/NL2SQL/evaluation_results/spider_nlq_evaluation_backup.json


## What the final numbers mean

For your **>94% target**, focus on:

```text
END-TO-END EXECUTION ACCURACY
```

because this evaluates:

```text
NLQ
 ↓
DB-ID prediction
 ↓
Schema Agent
 ↓
SQL generation
 ↓
validation/retry
 ↓
execution on predicted DB
```

Use the other numbers diagnostically:

```text
Top-10 DB recall
       ↓
If low → improve DB predictor.

Selected DB = gold
       ↓
If low despite high Top-10 → improve Schema Agent.

Oracle DB execution accuracy
       ↓
If low → improve SQL generation/reasoning.

End-to-end execution accuracy
       ↓
Your actual system result.
```

### Important

This notebook does **not** train the DB-ID model.

It loads the already-trained model from:

```text
MyDrive/NL2SQL/db_id_predictor/
```

and uses the Colab GPU for inference when CUDA is available.
